# Our 5 projects: CRG vs Module, per repo
One table per project, every kind: functions, classes, contains, inherits, imports, calls, files.
Each number is read from the saved run of its own notebook (named below) or recomputed here; nothing typed by hand.
- Reference = each language's own compiler/parser (javac, TypeScript compiler, CPython ast); files = `git ls-files`.
- **Found** = found / expected by the reference. **Correct** = correct / reported by the tool (the rest are wrong rows).
- Module = our tree-sitter module alone.

In [1]:
import pandas as pd, glob, pathlib, class_lab as L
R = L.ROOT / 'results'
P = {'guava': ('Java', 'java', 'iter-003_2026-11-17_2247'), 'spring-petclinic': ('Java', 'java', 'iter-002_2026-11-17_2008'),
     'crm-prototype': ('JavaScript', 'javascript', 'iter-003_2026-11-18_0743'), 'crm-prototype-client': ('JavaScript', 'javascript', 'iter-006_2026-11-18_0744'),
     'voice-assessment-agent': ('Python', 'python', 'iter-005_2026-11-17_2251')}
REF = {'java': 'javac', 'javascript': 'TypeScript compiler', 'python': 'CPython ast'}
rows = []
def add(p, kind, exp, cf, cr, mf, mr, ref):
    rows.append(dict(project=p, language=P[p][0], kind=kind, expected=exp, crg_found=cf, crg_reported=cr, mod_found=mf, mod_reported=mr, reference=ref))

## Functions — from `results/treesitter-module/2026-11-18_1555/named_scores.csv` (treesitter-module run). Correct is not stored for functions in that run.

In [2]:
f = pd.read_csv(R / 'treesitter-module/2026-11-18_1555/named_scores.csv')
for p in P:
    c = f[(f.app == p) & (f.tool == 'code-review-graph')].iloc[0]; m = f[(f.app == p) & (f.tool == 'module')].iloc[0]
    add(p, 'Functions', int(c.expected), int(c.found), None, int(m.found), None, REF[P[p][1]])

## Classes — recomputed with `class_lab.py` (same code as `class-crg-<project>.ipynb` / `class-module-<project>.ipynb`)

In [3]:
for p, (_, lang, run) in P.items():
    repo = L.ROOT / 'apps' / p; files = L.repo_files(repo, lang)
    ref = L.reference(repo, lang, files); ref = ref[ref.error.isna()]; named = ref[ref.kind != 'anonymous'].reset_index(drop=True)
    crg = L.crg_classes(R / p / 'runs' / run / 'crg/graph.db', repo, lang, files); mod = L.module(repo, lang, files)
    sc, _, _ = L.score(named, crg); sm, _, _ = L.score(named, mod)
    add(p, 'Classes', sc['expected'], sc['found'], sc['found'] + sc['extra'], sm['found'], sm['found'] + sm['extra'], REF[lang])

## Edges — newest `results/<project>/checks/edges_*/summary.csv` (from `edge-check-<project>.ipynb`)

In [4]:
names = {'CONTAINS': 'Contains', 'INHERITS': 'Inherits', 'IMPORTS_FROM': 'Imports', 'CALLS': 'Calls'}
for p in P:
    s = pd.read_csv(sorted(glob.glob(str(R / p / 'checks/edges_*/summary.csv')))[-1]); s.columns = ['edge', 'expected', 'reported', 'matched', 'missed', 'extra', 'r', 'pr']
    for e, k in names.items():
        c = s[s.edge == e].iloc[0]; m = s[s.edge.str.replace('_', ' ').str.strip() == f'{e} (module)'.replace('_', ' ')].iloc[0]
        add(p, k, int(c.expected), int(c.matched), int(c.reported), int(m.matched), int(m.reported), REF[P[p][1]])

## Files — newest `results/<project>/checks/extras_*/summary.csv` (from `extras-check-5repos.ipynb`). File nodes are made by CRG only.

In [5]:
for p in P:
    s = pd.read_csv(sorted(glob.glob(str(R / p / 'checks/extras_*/summary.csv')))[-1])
    for k, lab in [('FILE', 'Files'), ('JSON files', 'JSON files'), ('YAML files', 'YAML files')]:
        c = s[s.kind == k].iloc[0]
        add(p, lab, int(c.expected), int(c.matched), int(c.CRG), None, None, 'git ls-files')

In [6]:
t = pd.DataFrame(rows)
pct = lambda a, b: round(100 * a / b, 1) if b else None
t['crg_found_pct'] = [pct(a, b) for a, b in zip(t.crg_found, t.expected)]
t['crg_correct_pct'] = [pct(a, b) if pd.notna(b) else None for a, b in zip(t.crg_found, t.crg_reported)]
t['mod_found_pct'] = [pct(a, b) if pd.notna(a) else None for a, b in zip(t.mod_found, t.expected)]
t['mod_correct_pct'] = [pct(a, b) if pd.notna(b) else None for a, b in zip(t.mod_found, t.mod_reported)]
t.to_csv(R / 'five-projects-crg-vs-module.csv', index=False)
t

,project,language,kind,expected,crg_found,crg_reported,mod_found,mod_reported,reference,crg_found_pct,crg_correct_pct,mod_found_pct,mod_correct_pct
0,guava,Java,Functions,59236,59188,NaN,59236.0,NaN,javac,99.9,NaN,100.0,NaN
1,spring-petclinic,Java,Functions,195,192,NaN,195.0,NaN,javac,98.5,NaN,100.0,NaN
2,crm-prototype,JavaScript,Functions,227,206,NaN,227.0,NaN,TypeScript compiler,90.7,NaN,100.0,NaN
3,crm-prototype-client,JavaScript,Functions,623,559,NaN,623.0,NaN,TypeScript compiler,89.7,NaN,100.0,NaN
4,voice-assessment-agent,Python,Functions,487,487,NaN,487.0,NaN,CPython ast,100.0,NaN,100.0,NaN
5,guava,Java,Classes,7400,7260,7260.0,7400.0,7400.0,javac,98.1,100.0,100.0,100.0
6,spring-petclinic,Java,Classes,50,50,50.0,50.0,50.0,javac,100.0,100.0,100.0,100.0
7,crm-prototype,JavaScript,Classes,6,6,6.0,6.0,6.0,TypeScript compiler,100.0,100.0,100.0,100.0
8,crm-prototype-client,JavaScript,Classes,1,1,1.0,1.0,1.0,TypeScript compiler,100.0,100.0,100.0,100.0
9,voice-assessment-agent,Python,Classes,67,62,67.0,67.0,67.0,CPython ast,92.5,92.5,100.0,100.0
